<a href="https://colab.research.google.com/github/AnujikaSasindi/sri-lanka-tourism-sentiment-analysis/blob/Main/Try_Again_1__Dataset_processsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Removing unwanted columns from main dataset**

In [ ]:
import pandas as pd

# 1. Upload your file (run this cell, then choose your file from your computer)
from google.colab import files
uploaded = files.upload()

# 2. Load the dataset - change 'your_file.csv' to your actual filename
df = pd.read_excel('Reviews.xlsx')

print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Saving Reviews.xlsx to Reviews.xlsx
Shape of dataset: (16156, 14)

Column names:
['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID', 'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date', 'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text']

First 5 rows:


,Location_Name,Located_City,Location,Location_Type,User_ID,User_Location,User_Locale,User_Contributions,Travel_Date,Published_Date,Rating,Helpful_Votes,Title,Text
0,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 1,"Dunsborough, Australia",en_US,8,2019-07,2019-07-31T07:53:21-04:00,5,1,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...
1,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 2,"Bendigo, Australia",en_US,4,2019-06,2019-07-21T21:50:11-04:00,4,0,Best for surfing,"Overall, it is a wonderful experience. We visi..."
2,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 3,"Melbourne, Australia",en_US,13,2019-07,2019-07-15T18:52:55-04:00,5,0,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h..."
3,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 4,"Ericeira, Portugal",en_US,4,2019-06,2019-07-03T10:32:41-04:00,5,0,Sun and waves.,Good place for surf and a few stores to going ...
4,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 5,"Pistoia, Italy",en_US,14,2019-07,2019-07-02T17:07:02-04:00,5,0,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...


In [ ]:
columns_to_drop = ['Location', 'User_ID', 'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date', 'Published_Date', 'Helpful_Votes']

df_cleaned = df.drop(columns=columns_to_drop)

# 5. Verify it worked
print("New shape:", df_cleaned.shape)
print("\nRemaining columns:")
print(df_cleaned.columns.tolist())
df_cleaned.head()

New shape: (16156, 6)

Remaining columns:
['Location_Name', 'Located_City', 'Location_Type', 'Rating', 'Title', 'Text']


,Location_Name,Located_City,Location_Type,Rating,Title,Text
0,Arugam Bay,Arugam Bay,Beaches,5,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...
1,Arugam Bay,Arugam Bay,Beaches,4,Best for surfing,"Overall, it is a wonderful experience. We visi..."
2,Arugam Bay,Arugam Bay,Beaches,5,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h..."
3,Arugam Bay,Arugam Bay,Beaches,5,Sun and waves.,Good place for surf and a few stores to going ...
4,Arugam Bay,Arugam Bay,Beaches,5,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...


In [ ]:
# 6. Save the cleaned dataset
df_cleaned.to_excel('columns_removed_cleaned_dataset_1.xlsx', index=False)

# Optional: download it to your computer
files.download('columns_removed_cleaned_dataset_1.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Rating to Sentiment labelling**

In [ ]:
# Create sentiment label based on rating
def rating_to_sentiment(Rating):
    if Rating in [1, 2]:
        return 'Negative'
    elif Rating == 3:
        return 'Neutral'
    elif Rating in [4, 5]:
        return 'Positive'
    else:
        return None

df_cleaned['Sentiment'] = df_cleaned['Rating'].apply(rating_to_sentiment)

# Check the result
print(df_cleaned[['Rating', 'Sentiment']].head(10))

print("\nSentiment distribution:")
print(df_cleaned['Sentiment'].value_counts())
df_cleaned.head()

   Rating Sentiment
0       5  Positive
1       4  Positive
2       5  Positive
3       5  Positive
4       5  Positive
5       5  Positive
6       5  Positive
7       5  Positive
8       5  Positive
9       1  Negative

Sentiment distribution:
Sentiment
Positive    12845
Neutral      2166
Negative     1145
Name: count, dtype: int64


,Location_Name,Located_City,Location_Type,Rating,Title,Text,Sentiment
0,Arugam Bay,Arugam Bay,Beaches,5,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...,Positive
1,Arugam Bay,Arugam Bay,Beaches,4,Best for surfing,"Overall, it is a wonderful experience. We visi...",Positive
2,Arugam Bay,Arugam Bay,Beaches,5,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h...",Positive
3,Arugam Bay,Arugam Bay,Beaches,5,Sun and waves.,Good place for surf and a few stores to going ...,Positive
4,Arugam Bay,Arugam Bay,Beaches,5,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...,Positive


In [ ]:
summary = pd.DataFrame({
    'Count': df_cleaned['Sentiment'].value_counts(),
    'Percentage': (df_cleaned['Sentiment'].value_counts(normalize=True) * 100).round(2)
})
print(summary)

           Count  Percentage
Sentiment                   
Positive   12845       79.51
Neutral     2166       13.41
Negative    1145        7.09


**Attraction Categories defining by clustering**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

# 1. Combine the relevant columns into one text field for clustering
df_cleaned['Location_Name'] = df_cleaned['Location_Name'].fillna('')
df_cleaned['Location_Type'] = df_cleaned['Location_Type'].fillna('')
df_cleaned['Review_Text'] = df_cleaned['Review_Text'].fillna('')

df_cleaned['Cluster_Text'] = (
    df_cleaned['Location_Name'] + ' ' +
    df_cleaned['Location_Type'] + ' ' +
    df_cleaned['Review_Text']
)

print(df_cleaned['Cluster_Text'].head())

0    Arugam Bay Beaches Best nail spa in Arugam bay...
1    Arugam Bay Beaches Best for surfing Overall, i...
2    Arugam Bay Beaches We Love Arugam Bay Great pl...
3    Arugam Bay Beaches Sun and waves. Good place f...
4    Arugam Bay Beaches Great swimming, surfing, gr...
Name: Cluster_Text, dtype: object


In [ ]:
df_cleaned = df_cleaned.drop(columns=['cluster_text', 'Attraction_Cluster', 'Attraction_Category'], errors='ignore')

print(df_cleaned.columns.tolist())

['Location_Name', 'Located_City', 'Location_Type', 'Title', 'Text', 'Review_Text', 'Rating', 'Sentiment', 'Cluster_Text']


In [ ]:
df_cleaned.head()

,Location_Name,Located_City,Location_Type,Title,Text,Review_Text,Rating,Sentiment,Cluster_Text
0,Arugam Bay,Arugam Bay,Beaches,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...,Best nail spa in Arugam bay on the water! I h...,5,Positive,Arugam Bay Beaches Best nail spa in Arugam bay...
1,Arugam Bay,Arugam Bay,Beaches,Best for surfing,"Overall, it is a wonderful experience. We visi...","Best for surfing Overall, it is a wonderful ex...",4,Positive,"Arugam Bay Beaches Best for surfing Overall, i..."
2,Arugam Bay,Arugam Bay,Beaches,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h...","We Love Arugam Bay Great place to chill, swim,...",5,Positive,Arugam Bay Beaches We Love Arugam Bay Great pl...
3,Arugam Bay,Arugam Bay,Beaches,Sun and waves.,Good place for surf and a few stores to going ...,Sun and waves. Good place for surf and a few s...,5,Positive,Arugam Bay Beaches Sun and waves. Good place f...
4,Arugam Bay,Arugam Bay,Beaches,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...,"Great swimming, surfing, great fish aznd frien...",5,Positive,"Arugam Bay Beaches Great swimming, surfing, gr..."


In [ ]:
# 2. TF-IDF vectorize
# Using ngram_range=(1,2) to capture short phrases like "sea turtle" or "rock temple"
tfidf = TfidfVectorizer(max_features=1000, stop_words='english', min_df=5, ngram_range=(1, 2))
X_attraction = tfidf.fit_transform(df_cleaned['Cluster_Text'])

print("TF-IDF matrix shape:", X_attraction.shape)

TF-IDF matrix shape: (16156, 1000)


In [ ]:
# 3. Cluster into 5 groups
n_clusters = 6
kmeans_attraction = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_cleaned['Attraction_Cluster'] = kmeans_attraction.fit_predict(X_attraction)

print(df_cleaned['Attraction_Cluster'].value_counts().sort_index())

Attraction_Cluster
0    1977
1    3015
2    2768
3    2739
4    2062
5    3595
Name: count, dtype: int64


In [ ]:
# 4. Extract top key terms per cluster (this is how you'll name them)
feature_names = np.array(tfidf.get_feature_names_out())
top_n = 15

for c in range(n_clusters):
    center = kmeans_attraction.cluster_centers_[c]
    top_indices = center.argsort()[::-1][:top_n]
    top_terms = feature_names[top_indices]
    print(f"\n--- Cluster {c} (n={sum(df_cleaned['Attraction_Cluster']==c)}) ---")
    print(", ".join(top_terms))


--- Cluster 0 (n=1977) ---
beach, beaches, beach beaches, clean, nice, sea, beautiful, bentota, sand, mirissa, waves, bentota beach, negombo, water, good

--- Cluster 1 (n=3015) ---
temple, religious, religious sites, sites, temple religious, place, stupa, visit, buddha, buddhist, sacred, sri, maha, tree, vihara

--- Cluster 2 (n=2768) ---
national, park, elephants, wildlife, national park, nature, nature wildlife, wildlife areas, areas, parks, national parks, park national, safari, elephant, birds

--- Cluster 3 (n=2739) ---
museum, fort, historic, museums, historic sites, museum museums, sigiriya, sites, rock, history, visit, tsunami, worth, fort historic, place

--- Cluster 4 (n=2062) ---
tea, factory, tour, tea factory, farms, factory farms, interesting, guide, process, estate, visit, plantation, cup, informative, shop

--- Cluster 5 (n=3595) ---
gardens, lake, falls, garden, water, waterfalls, waterfall, bodies water, bodies, nice, place, beautiful, falls waterfalls, kandy, walk


In [ ]:
# 5. Optional: cross-check against actual Location_Type to validate the clusters
pd.crosstab(df_cleaned['Attraction_Cluster'], df_cleaned['Location_Type']).T

Attraction_Cluster,0,1,2,3,4,5
Location_Type,,,,,,
Beaches,1976,0,5,0,1,128
Bodies of Water,0,0,1,1,0,837
Farms,0,0,0,1,1745,138
Gardens,0,0,0,0,1,1353
Historic Sites,0,1,1,1472,45,0
Museums,0,0,0,1263,262,0
National Parks,1,0,1204,0,0,0
Nature & Wildlife Areas,0,0,1553,0,0,4
Religious Sites,0,3014,0,2,0,1


In [ ]:
cluster_names = {
    0: 'Beaches & Coastal Areas',
    1: 'Temples & Religious Sites',
    2: 'Wildlife & National Parks',
    3: 'Museums & Historic Sites',
    4: 'Tea Plantations & Factory Tours',
    5: 'Gardens, Lakes & Waterfalls'
}

df_cleaned['Attraction_Category'] = df_cleaned['Attraction_Cluster'].map(cluster_names)

# Quick check
print(df_cleaned[['Attraction_Cluster', 'Attraction_Category']].head(10))
print("\nValue counts:")
print(df_cleaned['Attraction_Category'].value_counts())

   Attraction_Cluster          Attraction_Category
0                   5  Gardens, Lakes & Waterfalls
1                   0      Beaches & Coastal Areas
2                   5  Gardens, Lakes & Waterfalls
3                   0      Beaches & Coastal Areas
4                   5  Gardens, Lakes & Waterfalls
5                   5  Gardens, Lakes & Waterfalls
6                   5  Gardens, Lakes & Waterfalls
7                   0      Beaches & Coastal Areas
8                   5  Gardens, Lakes & Waterfalls
9                   5  Gardens, Lakes & Waterfalls

Value counts:
Attraction_Category
Gardens, Lakes & Waterfalls        3595
Temples & Religious Sites          3015
Wildlife & National Parks          2768
Museums & Historic Sites           2739
Tea Plantations & Factory Tours    2062
Beaches & Coastal Areas            1977
Name: count, dtype: int64


In [ ]:
df_cleaned = df_cleaned.drop(columns=['attraction_cluster'], errors='ignore')

print(df_cleaned.columns.tolist())

['Location_Name', 'Located_City', 'Location_Type', 'Title', 'Text', 'Review_Text', 'Rating', 'Sentiment', 'cluster_text', 'Attraction_Category', 'Attraction_Cluster']


In [ ]:
df_cleaned.head()

,Location_Name,Located_City,Location_Type,Title,Text,Review_Text,Rating,Sentiment,Cluster_Text,Attraction_Cluster,Attraction_Category
0,Arugam Bay,Arugam Bay,Beaches,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...,Best nail spa in Arugam bay on the water! I h...,5,Positive,Arugam Bay Beaches Best nail spa in Arugam bay...,5,"Gardens, Lakes & Waterfalls"
1,Arugam Bay,Arugam Bay,Beaches,Best for surfing,"Overall, it is a wonderful experience. We visi...","Best for surfing Overall, it is a wonderful ex...",4,Positive,"Arugam Bay Beaches Best for surfing Overall, i...",0,Beaches & Coastal Areas
2,Arugam Bay,Arugam Bay,Beaches,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h...","We Love Arugam Bay Great place to chill, swim,...",5,Positive,Arugam Bay Beaches We Love Arugam Bay Great pl...,5,"Gardens, Lakes & Waterfalls"
3,Arugam Bay,Arugam Bay,Beaches,Sun and waves.,Good place for surf and a few stores to going ...,Sun and waves. Good place for surf and a few s...,5,Positive,Arugam Bay Beaches Sun and waves. Good place f...,0,Beaches & Coastal Areas
4,Arugam Bay,Arugam Bay,Beaches,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...,"Great swimming, surfing, great fish aznd frien...",5,Positive,"Arugam Bay Beaches Great swimming, surfing, gr...",5,"Gardens, Lakes & Waterfalls"


In [ ]:
#Save the cleaned dataset (sentiment + attarction category)
df_cleaned.to_excel('cleaned_dataset_1_(sentiment + attarction category).xlsx', index=False)

# Optional: download it to your computer
files.download('cleaned_dataset_1_(sentiment + attarction category).xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Dataset Balancing with positive review clustering**

In [ ]:
df_cleaned.head()

# Review_Text ===> Title + Text
# Cluster_Text ===> Location_Name + Location_Type + Review_

,Location_Name,Located_City,Location_Type,Title,Text,Review_Text,Rating,Sentiment,Cluster_Text,Attraction_Cluster,Attraction_Category
0,Arugam Bay,Arugam Bay,Beaches,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...,Best nail spa in Arugam bay on the water! I h...,5,Positive,Arugam Bay Beaches Best nail spa in Arugam bay...,5,"Gardens, Lakes & Waterfalls"
1,Arugam Bay,Arugam Bay,Beaches,Best for surfing,"Overall, it is a wonderful experience. We visi...","Best for surfing Overall, it is a wonderful ex...",4,Positive,"Arugam Bay Beaches Best for surfing Overall, i...",0,Beaches & Coastal Areas
2,Arugam Bay,Arugam Bay,Beaches,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h...","We Love Arugam Bay Great place to chill, swim,...",5,Positive,Arugam Bay Beaches We Love Arugam Bay Great pl...,5,"Gardens, Lakes & Waterfalls"
3,Arugam Bay,Arugam Bay,Beaches,Sun and waves.,Good place for surf and a few stores to going ...,Sun and waves. Good place for surf and a few s...,5,Positive,Arugam Bay Beaches Sun and waves. Good place f...,0,Beaches & Coastal Areas
4,Arugam Bay,Arugam Bay,Beaches,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...,"Great swimming, surfing, great fish aznd frien...",5,Positive,"Arugam Bay Beaches Great swimming, surfing, gr...",5,"Gardens, Lakes & Waterfalls"


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np

# 1. Isolate positive reviews
positive_df = df_cleaned[df_cleaned['Sentiment'] == 'Positive'].copy()
print("Positive reviews count:", len(positive_df))

# 2. TF-IDF vectorize the review text
tfidf = TfidfVectorizer(max_features=500, stop_words='english', min_df=5)
X_tfidf = tfidf.fit_transform(positive_df['Review_Text'])

print("TF-IDF matrix shape:", X_tfidf.shape)

Positive reviews count: 12845
TF-IDF matrix shape: (12845, 500)


In [ ]:
# 3. Cluster into groups (try 15 first, adjust later based on results)
n_clusters = 15
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
positive_df['cluster'] = kmeans.fit_predict(X_tfidf)

# See how many reviews landed in each cluster
print(positive_df['cluster'].value_counts().sort_index())

cluster
0     1474
1     1234
2      376
3      504
4      385
5      610
6      283
7      750
8      666
9      398
10    1014
11    3541
12     958
13     303
14     349
Name: count, dtype: int64


In [ ]:
# 4. Compute distance of each point to its own cluster centroid
distances = kmeans.transform(X_tfidf)  # distance to every centroid
positive_df['dist_to_centroid'] = distances[np.arange(len(positive_df)), positive_df['cluster']]

# 5. Within each cluster, drop the top X% farthest points (outliers)
def remove_outliers_per_cluster(group, drop_frac=0.1):
    threshold = group['dist_to_centroid'].quantile(1 - drop_frac)
    return group[group['dist_to_centroid'] <= threshold]

positive_cleaned = positive_df.groupby('cluster', group_keys=False).apply(
    lambda g: remove_outliers_per_cluster(g, drop_frac=0.10)
)
print("Before outlier removal:", len(positive_df))
print("After outlier removal:", len(positive_cleaned))

Before outlier removal: 12845
After outlier removal: 11555


/tmp/ipykernel_1703/3921470469.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  positive_cleaned = positive_df.groupby('cluster', group_keys=False).apply(


In [ ]:
# 6. Set target positive count as a percentage of the ORIGINAL positive count
target_keep_fraction = 0.30  # adjust this to whatever % you decide on

target_positive_count = int(len(positive_df) * target_keep_fraction)
print(f"Original positive count: {len(positive_df)}")
print(f"Target positive count ({target_keep_fraction*100:.0f}%): {target_positive_count}")

Original positive count: 12845
Target positive count (30%): 3853


In [ ]:
# 7. Proportionally sample down to target count, drawing from the outlier-cleaned set
sampled_positive = positive_cleaned.groupby('cluster', group_keys=False).apply(
    lambda g: g.sample(frac=min(1.0, target_positive_count / len(positive_cleaned)), random_state=42)
)

print("Final sampled positive count:", len(sampled_positive))
print(sampled_positive['cluster'].value_counts().sort_index())

Final sampled positive count: 3853
cluster
0      442
1      370
2      113
3      151
4      115
5      183
6       85
7      225
8      200
9      119
10     304
11    1063
12     287
13      91
14     105
Name: count, dtype: int64


/tmp/ipykernel_1703/1111052805.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_positive = positive_cleaned.groupby('cluster', group_keys=False).apply(


In [ ]:
# 8. Recombine with neutral and negative to form your balanced dataset
neutral_df = df_cleaned[df_cleaned['Sentiment'] == 'Neutral']
negative_df = df_cleaned[df_cleaned['Sentiment'] == 'Negative']

balanced_df = pd.concat([
    sampled_positive.drop(columns=['cluster', 'dist_to_centroid'], errors='ignore'),
    neutral_df,
    negative_df
], ignore_index=True)

print(balanced_df['Sentiment'].value_counts())
print(balanced_df['Sentiment'].value_counts(normalize=True).round(4) * 100)

Sentiment
Positive    3853
Neutral     2166
Negative    1145
Name: count, dtype: int64
Sentiment
Positive    53.78
Neutral     30.23
Negative    15.98
Name: proportion, dtype: float64


In [ ]:
balanced_df.head()

,Location_Name,Located_City,Location_Type,Title,Text,Review_Text,Rating,Sentiment,Cluster_Text,Attraction_Cluster,Attraction_Category
0,Handunugoda Tea Estate,Ahangama,Farms,Wonderful Experience,This was our third tea factory tour during our...,Wonderful Experience This was our third tea fa...,5,Positive,Handunugoda Tea Estate Farms Wonderful Experie...,4,Tea Plantations & Factory Tours
1,Glenloch Tea Factory,Katukitula,Farms,In-TEA-resting,A worth visit for everyone who drinks tea. Gui...,In-TEA-resting A worth visit for everyone who ...,4,Positive,Glenloch Tea Factory Farms In-TEA-resting A wo...,4,Tea Plantations & Factory Tours
2,Handunugoda Tea Estate,Ahangama,Farms,"The best tea experience, great history","Hidden away in a beautiful stretch of green, t...","The best tea experience, great history Hidden ...",5,Positive,Handunugoda Tea Estate Farms The best tea expe...,4,Tea Plantations & Factory Tours
3,Bluefield Tea Gardens,Nuwara Eliya,Farms,Informative tour,We really enjoyed our tour at the Bluefield te...,Informative tour We really enjoyed our tour at...,5,Positive,Bluefield Tea Gardens Farms Informative tour W...,4,Tea Plantations & Factory Tours
4,Ceylon Tea Museum,Kandy,Museums,Under appreciated - well worth,We spend a few enjoyable hours here. It is aro...,Under appreciated - well worth We spend a few ...,4,Positive,Ceylon Tea Museum Museums Under appreciated - ...,4,Tea Plantations & Factory Tours


In [ ]:
#Save the cleaned dataset (sentiment + attarction category)
balanced_df.to_excel('balanced_cleaned_dataset_1_(sentiment + attarction category).xlsx', index=False)

# Optional: download it to your computer
files.download('balanced_cleaned_dataset_1_(sentiment + attarction category).xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>